In [1]:
from pathlib import Path
import shutil
import subprocess

# 1. Xác định thư mục dự án
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebook":
    PROJECT_DIR = PROJECT_DIR.parent

# 2. Khai báo dữ liệu gốc
DATA_DIR = PROJECT_DIR / "data" / "ACME_Insurance"

OWL_FILE = DATA_DIR / "ontology" / "insurance.ttl"
R2RML_FILE = DATA_DIR / "data" / "data.world_P&C_Insurance_Ontology_V1.r2rml"
DB_FILE = PROJECT_DIR / "results" / "sql" / "acme.duckdb"

# 3. Kiểm tra các file đầu vào
for name, path in [
    ("OWL", OWL_FILE),
    ("R2RML", R2RML_FILE),
    ("DuckDB", DB_FILE)
]:
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

    assert path.is_file(), f"File not found: {path}"

# 4. Kiểm tra Docker và Java
print("\n=== ONTOP ENVIRONMENT ===")

print("Java:", "Installed" if shutil.which("java") else "Not installed")
print("Docker:", "Installed" if shutil.which("docker") else "Not installed")

if shutil.which("docker"):
    result = subprocess.run(
        ["docker", "info", "--format", "{{.ServerVersion}}"],
        capture_output=True,
        text=True,
        timeout=15
    )

    print(
        "Docker Engine:",
        result.stdout.strip() if result.returncode == 0 else "Not running"
    )


OWL: OK
R2RML: OK
DuckDB: OK

=== ONTOP ENVIRONMENT ===
Java: Installed
Docker: Installed
Docker Engine: Not running


In [2]:
# 1. Kiểm tra phiên bản Java
java = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

assert java.returncode == 0
print("Java:", java.stderr.splitlines()[0])

# 2. Tạo thư mục cấu hình Ontop
ONTOP_DIR = PROJECT_DIR / "results" / "ontop"
ONTOP_DIR.mkdir(parents=True, exist_ok=True)

# 3. Tạo file kết nối DuckDB
PROPERTIES_FILE = ONTOP_DIR / "duckdb.properties"

PROPERTIES_FILE.write_text(
    f"jdbc.url=jdbc:duckdb:{DB_FILE.resolve().as_posix()}\n"
    "jdbc.driver=org.duckdb.DuckDBDriver\n",
    encoding="utf-8"
)

# 4. Kiểm tra kết quả
print("\n=== ONTOP CONFIGURATION ===")
print("Database:", DB_FILE.name)
print("JDBC Driver: org.duckdb.DuckDBDriver")
print("Properties:", PROPERTIES_FILE)
print("Status: Ready")

Java: openjdk version "11.0.28" 2025-07-15

=== ONTOP CONFIGURATION ===
Database: acme.duckdb
JDBC Driver: org.duckdb.DuckDBDriver
Properties: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\ontop\duckdb.properties
Status: Ready


In [3]:
# 1. Xác định vị trí Ontop CLI
ONTOP_HOME = Path.home() / "tools" / "ontop"
ontop_files = list(ONTOP_HOME.rglob("ontop.bat"))

assert ontop_files, "Không tìm thấy ontop.bat"

ONTOP_BAT = ontop_files[0]

# 2. Kiểm tra phiên bản Ontop
result = subprocess.run(
    [str(ONTOP_BAT), "--version"],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    timeout=30
)

# 3. Hiển thị kết quả
assert result.returncode == 0, result.stderr

print("=== ONTOP CLI ===")
print("Path:", ONTOP_BAT)
print("Version:", result.stdout.strip())
print("Status: Ready")

=== ONTOP CLI ===
Path: C:\Users\Win 11 New\tools\ontop\ontop.bat
Version: ontop version 5.5.0
Status: Ready


In [10]:
import duckdb
import json

# 1. Tạo database tương thích với JDBC 1.4
COMPAT_DB = ONTOP_DIR / "acme_ontop.duckdb"

if not COMPAT_DB.exists():
    conn = duckdb.connect()

    try:
        conn.execute(
            f"ATTACH '{DB_FILE.as_posix()}' AS source (READ_ONLY)"
        )
        conn.execute(
            f"ATTACH '{COMPAT_DB.as_posix()}' AS target "
            "(STORAGE_VERSION 'v1.4.0')"
        )

        # Sao chép toàn bộ bảng và dữ liệu
        conn.execute("COPY FROM DATABASE source TO target")

    finally:
        conn.close()

# 2. Cập nhật cấu hình Ontop
# Cấu hình kết nối và suy luận kiểu dữ liệu
PROPERTIES_FILE.write_text(
    f"jdbc.url=jdbc:duckdb:{COMPAT_DB.as_posix()}\n"
    "jdbc.driver=org.duckdb.DuckDBDriver\n"
    "ontop.allowRetrievingBlackBoxViewMetadataFromDB=true\n",
    encoding="utf-8"
)

# 3. Kiểm tra kết nối Ontop
METADATA_FILE = ONTOP_DIR / "db-metadata.json"

result = subprocess.run(
    [
        str(ONTOP_BAT),
        "extract-db-metadata",
        "-p", PROPERTIES_FILE.resolve().as_uri(),
        "-o", str(METADATA_FILE)
    ],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=120
)

# 4. Hiển thị kết quả
if result.returncode != 0:
    print(result.stdout[-1500:])
    print(result.stderr[-2500:])
    raise RuntimeError("Ontop connection failed")

metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))

print("=== ONTOP DATABASE CONNECTION ===")
print("Original database:", DB_FILE.name)
print("Compatible database:", COMPAT_DB.name)
print("Relations:", len(metadata["relations"]))
print("Status: OK")

=== ONTOP DATABASE CONNECTION ===
Original database: acme.duckdb
Compatible database: acme_ontop.duckdb
Relations: 29
Status: OK


In [20]:
from rdflib import Graph, Namespace, RDF
import shutil

# 1. Tạo bản mapping tương thích với DuckDB local
MAPPING_FILE = ONTOP_DIR / "acme_mapping.ttl"

mapping_text = R2RML_FILE.read_text(encoding="utf-8")

# Loại tiền tố tên bảng riêng của data.world
old_prefix = 'rr:tableName "myinsurancecompany.omg-pc-database.'
new_prefix = 'rr:tableName "'

assert mapping_text.count(old_prefix) == 23

# Chuẩn hóa tên bảng cho DuckDB
mapping_text = mapping_text.replace(old_prefix, new_prefix)

# Sửa liên kết Policy Holder không nhất quán trong R2RML gốc
old_uri = "/PolicyHolder-{party_identifier}"
new_uri = "/Policy-Holder-{party_identifier}"

assert mapping_text.count(old_uri) == 1
mapping_text = mapping_text.replace(old_uri, new_uri)

# Lưu bản mapping tương thích local
MAPPING_FILE.write_text(mapping_text, encoding="utf-8")

print("Policy Holder URI fixed: 1")

print("Table references adapted: 23")

# 2. Đếm số TriplesMaps trong mapping gốc
mapping = Graph()
mapping.parse(str(MAPPING_FILE), format="turtle")

RR = Namespace("http://www.w3.org/ns/r2rml#")
triples_maps = set(mapping.subjects(RDF.type, RR.TriplesMap))

print("R2RML TriplesMaps:", len(triples_maps))
assert len(triples_maps) == 62

import duckdb

# Chuẩn hóa các cột ngày theo OWL Ontology gốc
date_columns = {
    "claim": ["claim_open_date", "claim_close_date"],
    "policy": ["effective_date", "expiration_date"],
    "policy_coverage_detail": ["effective_date", "expiration_date"]
}

with duckdb.connect(str(COMPAT_DB)) as conn:
    for table, columns in date_columns.items():
        types = {
            row[0].lower(): row[1].upper()
            for row in conn.execute(f"DESCRIBE {table}").fetchall()
        }

        for column in columns:
            if types[column] == "DATE":
                conn.execute(
                    f"ALTER TABLE {table} "
                    f"ALTER COLUMN {column} TYPE TIMESTAMP"
                )
                print(f"Converted: {table}.{column}")

print("Date compatibility: Ready")

# 3. Kiểm tra mapping bằng Ontop
result = subprocess.run(
    [
        str(ONTOP_BAT),
        "validate",
        "-m", MAPPING_FILE.resolve().as_uri(),
        "-t", str(OWL_FILE.resolve()),
        "-p", PROPERTIES_FILE.resolve().as_uri()
    ],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=180
)

# 4. Hiển thị kết quả kiểm tra
print("\n=== ONTOP MAPPING VALIDATION ===")
print("Exit code:", result.returncode)
print("Output:", result.stdout[-2000:])
print("Errors:", result.stderr[-3000:])

if result.returncode == 0:
    print("Mapping validation: OK")
else:
    print("Mapping validation: FAILED")

Policy Holder URI fixed: 1
Table references adapted: 23
R2RML TriplesMaps: 62
Date compatibility: Ready

=== ONTOP MAPPING VALIDATION ===
Exit code: 0
Output: Validation completed

Errors: 
Mapping validation: OK


In [13]:
import pandas as pd

# 1. Tạo truy vấn SPARQL kiểm tra
QUERY_FILE = ONTOP_DIR / "test_claim_count.rq"
RESULT_FILE = ONTOP_DIR / "test_claim_count.csv"

QUERY_FILE.write_text(
    "PREFIX in: <http://data.world/schema/insurance/>\n"
    "SELECT (COUNT(DISTINCT ?claim) AS ?total)\n"
    "WHERE { ?claim a in:Claim . }",
    encoding="utf-8"
)

# 2. Thực thi SPARQL bằng Ontop
result = subprocess.run(
    [
        str(ONTOP_BAT), "query",
        "-m", MAPPING_FILE.resolve().as_uri(),
        "-t", str(OWL_FILE.resolve()),
        "-p", PROPERTIES_FILE.resolve().as_uri(),
        "-q", str(QUERY_FILE),
        "-o", str(RESULT_FILE)
    ],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=180
)

# 3. Kiểm tra lỗi thực thi
if result.returncode != 0:
    print(result.stdout[-1500:])
    print(result.stderr[-2500:])
    raise RuntimeError("SPARQL execution failed")

# 4. Đọc kết quả từ Ontop
sparql_result = pd.read_csv(RESULT_FILE)
sparql_count = int(sparql_result.iloc[0, 0])

# 5. So sánh với dữ liệu SQL
with duckdb.connect(str(COMPAT_DB), read_only=True) as conn:
    sql_count = conn.execute(
        "SELECT COUNT(*) FROM claim"
    ).fetchone()[0]

print("=== VIRTUAL KG TEST ===")
print("SPARQL Claim Count:", sparql_count)
print("SQL Claim Count:", sql_count)
print("Results Match:", sparql_count == sql_count)

=== VIRTUAL KG TEST ===
SPARQL Claim Count: 2
SQL Claim Count: 2
Results Match: True


In [14]:
import re
from rdflib import Literal

# 1. Đọc benchmark gốc
benchmark = Graph()
benchmark.parse(
    str(DATA_DIR / "investigation" / "acme-benchmark.ttl"),
    format="turtle"
)

QA = Namespace("http://models.data.world/benchmarks/QandA#")
DWT = Namespace("https://templates.data.world/")

# 2. Lấy SQL và SPARQL Reference của tác giả
question = "How many claims do we have?"
inquiry = next(benchmark.subjects(QA.prompt, Literal(question)))

references = list(benchmark.objects(inquiry, QA.expects))

sql_ref = next(
    ref for ref in references
    if (ref, RDF.type, DWT.SqlQuery) in benchmark
)

sparql_ref = next(
    ref for ref in references
    if (ref, RDF.type, DWT.SparqlQuery) in benchmark
)

sql_query = str(benchmark.value(sql_ref, QA.queryText))
sparql_query = str(benchmark.value(sparql_ref, QA.queryText))

# 3. Chuyển SERVICE :mapped sang truy vấn Ontop local
local_query, changes = re.subn(
    r"\bSERVICE\s+[\w-]*:mapped\s*\{",
    "{",
    sparql_query,
    flags=re.IGNORECASE
)

assert changes == 1

# 4. Lưu và thực thi SPARQL Reference
QUERY_FILE = ONTOP_DIR / "reference_claim_count.rq"
RESULT_FILE = ONTOP_DIR / "reference_claim_count.csv"

QUERY_FILE.write_text(local_query, encoding="utf-8")

result = subprocess.run(
    [
        str(ONTOP_BAT), "query",
        "-m", MAPPING_FILE.resolve().as_uri(),
        "-t", str(OWL_FILE.resolve()),
        "-p", PROPERTIES_FILE.resolve().as_uri(),
        "-q", str(QUERY_FILE),
        "-o", str(RESULT_FILE)
    ],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=180
)

if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("SPARQL Reference failed")

# 5. So sánh với SQL Reference
sparql_count = int(pd.read_csv(RESULT_FILE).iloc[0, 0])

with duckdb.connect(str(COMPAT_DB), read_only=True) as conn:
    sql_count = conn.execute(sql_query).fetchone()[0]

print("=== ORIGINAL REFERENCE VALIDATION ===")
print("Question:", question)
print("SQL Reference:", sql_count)
print("SPARQL Reference:", sparql_count)
print("Results Match:", sql_count == sparql_count)

=== ORIGINAL REFERENCE VALIDATION ===
Question: How many claims do we have?
SQL Reference: 2
SPARQL Reference: 2
Results Match: True


In [21]:
import time
from rdflib import URIRef

# 1. Chuẩn bị thư mục kết quả
QUERY_DIR = ONTOP_DIR / "reference_queries"
RESULT_DIR = ONTOP_DIR / "reference_results"

QUERY_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

execution_log = []

# 2. Lấy 43 câu hỏi trực tiếp từ benchmark
inquiries = sorted(
    benchmark.subjects(RDF.type, QA.Inquiry),
    key=str
)

extra_question = "Return all the claims by claim number that were closed in 2019"

inquiries = [
    uri for uri in inquiries
    if str(benchmark.value(uri, QA.prompt)) != extra_question
]

assert len(inquiries) == 43

# 3. Duyệt 43 câu hỏi
for question_id, inquiry in enumerate(inquiries, start=1):

    # Lấy SPARQL Reference gốc
    ref = next(
        ref for ref in benchmark.objects(inquiry, QA.expects)
        if (ref, RDF.type, DWT.SparqlQuery) in benchmark
    )

    original = str(benchmark.value(ref, QA.queryText))

    # 3. Chuyển SERVICE của data.world sang Ontop local
    local, changes = re.subn(
        r"\bSERVICE\s+[\w-]*:mapped\s*(?=\{)",
        "",
        original,
        flags=re.IGNORECASE
    )

    assert changes == 1, f"Unexpected SERVICE in question {question_id}"

    original_file = QUERY_DIR / f"{question_id:02d}_original.rq"
    query_file = QUERY_DIR / f"{question_id:02d}_local.rq"
    result_file = RESULT_DIR / f"{question_id:02d}.csv"

    original_file.write_text(original, encoding="utf-8")
    query_file.write_text(local, encoding="utf-8")

    # Xóa kết quả cũ trước khi chạy lại
    result_file.unlink(missing_ok=True)

    # 4. Thực thi SPARQL trên Ontop
    start = time.perf_counter()

    try:
        result = subprocess.run(
            [
                str(ONTOP_BAT), "query",
                "-m", MAPPING_FILE.resolve().as_uri(),
                "-t", str(OWL_FILE.resolve()),
                "-p", PROPERTIES_FILE.resolve().as_uri(),
                "-q", str(query_file),
                "-o", str(result_file)
            ],
            cwd=ONTOP_BAT.parent,
            capture_output=True,
            text=True,
            errors="replace",
            timeout=180
        )

        if result.returncode == 0 and result_file.exists():
            rows = len(pd.read_csv(result_file))
            status = "OK"
            error = ""
        else:
            rows = None
            status = "ERROR"
            error = (result.stderr or result.stdout)[-1000:]

    except subprocess.TimeoutExpired:
        rows = None
        status = "TIMEOUT"
        error = "Execution exceeded 180 seconds"

    except Exception as e:
        rows = None
        status = "ERROR"
        error = str(e)

    # 5. Ghi log từng truy vấn
    execution_log.append({
        "ID": question_id,
        "Inquiry URI": str(inquiry),
        "Reference ID": str(ref),
        "Status": status,
        "Rows": rows,
        "Time (s)": round(time.perf_counter() - start, 2),
        "Error": error
    })

    pd.DataFrame(execution_log).to_csv(
        ONTOP_DIR / "reference_execution_log.csv",
        index=False
    )

    print(f"Question {question_id}/43: {status} | Rows: {rows}")

# 6. Tổng hợp kết quả
reference_log_df = pd.DataFrame(execution_log)

print("\n=== ONTOP REFERENCE VALIDATION ===")
print("Total:", len(reference_log_df))
print(reference_log_df["Status"].value_counts())
print("Empty results:", (reference_log_df["Rows"] == 0).sum())

print("\nFailed References:")
display(reference_log_df[reference_log_df["Status"] != "OK"])

Question 1/43: OK | Rows: 1
Question 2/43: OK | Rows: 2
Question 3/43: OK | Rows: 2
Question 4/43: OK | Rows: 1
Question 5/43: OK | Rows: 2
Question 6/43: OK | Rows: 1
Question 7/43: OK | Rows: 2
Question 8/43: OK | Rows: 2
Question 9/43: OK | Rows: 2
Question 10/43: OK | Rows: 2
Question 11/43: OK | Rows: 1
Question 12/43: OK | Rows: 2
Question 13/43: OK | Rows: 2
Question 14/43: OK | Rows: 6
Question 15/43: OK | Rows: 2
Question 16/43: OK | Rows: 2
Question 17/43: OK | Rows: 2
Question 18/43: ERROR | Rows: None
Question 19/43: OK | Rows: 1
Question 20/43: OK | Rows: 2
Question 21/43: OK | Rows: 1
Question 22/43: OK | Rows: 6
Question 23/43: OK | Rows: 6
Question 24/43: OK | Rows: 2
Question 25/43: OK | Rows: 1
Question 26/43: OK | Rows: 1
Question 27/43: OK | Rows: 1
Question 28/43: OK | Rows: 1
Question 29/43: OK | Rows: 2
Question 30/43: OK | Rows: 2
Question 31/43: OK | Rows: 1
Question 32/43: OK | Rows: 2
Question 33/43: OK | Rows: 2
Question 34/43: OK | Rows: 2
Question 35/43: O

,ID,Inquiry URI,Reference ID,Status,Rows,Time (s),Error
17,18,https://templates.data.world/IQ_6da3f7fcefcdd7...,https://templates.data.world/query-76d22258-22...,ERROR,NaN,5.77,F4JTupleExprTranslator.java:613)\n\tat it.unib...


In [19]:
# 1. Lấy danh sách SPARQL Reference trả về rỗng
empty_df = reference_log_df[
    (reference_log_df["Status"] == "OK") &
    (reference_log_df["Rows"] == 0)
].copy()

# 2. Đối chiếu số dòng với SQL Reference đã lưu
SQL_REF_DIR = PROJECT_DIR / "results" / "sql" / "reference_results"

def get_sql_rows(inquiry):
    refs = [
        ref for ref in benchmark.objects(inquiry, QA.expects)
        if (ref, RDF.type, DWT.SqlQuery) in benchmark
    ]

    return [
        len(pd.read_csv(SQL_REF_DIR / (str(ref).split("/")[-1] + ".csv")))
        for ref in refs
    ]

empty_df["Question"] = [
    str(benchmark.value(inquiries[int(i) - 1], QA.prompt))
    for i in empty_df["ID"]
]

empty_df["SQL Reference Rows"] = [
    get_sql_rows(inquiries[int(i) - 1])
    for i in empty_df["ID"]
]

print("=== EMPTY SPARQL REFERENCES ===")
display(empty_df[["ID", "Question", "SQL Reference Rows", "Rows"]])

# 3. Chạy lại riêng câu 18 để lấy thông báo lỗi đầy đủ
print("\n=== QUESTION 18 ERROR ===")

result = subprocess.run(
    [
        str(ONTOP_BAT), "query",
        "-m", MAPPING_FILE.resolve().as_uri(),
        "-t", str(OWL_FILE.resolve()),
        "-p", PROPERTIES_FILE.resolve().as_uri(),
        "-q", str(QUERY_DIR / "18_local.rq"),
        "-o", str(ONTOP_DIR / "debug18.csv")
    ],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=180
)

print("Exit code:", result.returncode)
print("Output:", result.stdout[:2000])
print("Error:", result.stderr[:4000])

=== EMPTY SPARQL REFERENCES ===


,ID,Question,SQL Reference Rows,Rows
0,1,What is the loss ratio of each policy and agen...,[1],0.0
2,3,"What are the loss payment, loss reserve, expen...",[2],0.0
6,7,"What are the total loss, which is the sum of l...",[2],0.0
7,8,What is the total amount of premiums that a po...,[2],0.0
8,9,"What are the loss payment, loss reserve, expen...",[2],0.0
13,14,What are all the premiums that have been paid ...,[6],0.0
23,24,Return policy holders and the claims they have...,[2],0.0
25,26,"What is the loss ratio, number of claims, tot...",[1],0.0
26,27,What is the total amount of premiums that a po...,[1],0.0
30,31,How many policies does each policy holder have...,[1],0.0



=== QUESTION 18 ERROR ===
Exit code: 0
Output: 00:15:05.818 |-INFO  in i.u.i.o.a.impl.OntopQueryEngineImpl - Ontop has completed the setup and it is ready for query answering!
00:15:05.820 |-INFO  in i.u.i.o.r.r.i.OntopVirtualRepository - Ontop virtual repository initialized successfully!
00:15:06.013 |-WARN  in i.u.i.o.a.r.impl.QuestQueryProcessor - Unexpected exception: it.unibz.inf.ontop.exception.OntopUnsupportedKGQueryException: The expression FunctionCall (http://data.world/function/functions#date_diff)
   Var (name=ClaimOpenDate)
   Var (name=ClaimCloseDate)
   ValueConstant (value="day")
 is not supported yet!
java.lang.RuntimeException: it.unibz.inf.ontop.exception.OntopUnsupportedKGQueryException: The expression FunctionCall (http://data.world/function/functions#date_diff)
   Var (name=ClaimOpenDate)
   Var (name=ClaimCloseDate)
   ValueConstant (value="day")
 is not supported yet!
	at it.unibz.inf.ontop.query.translation.impl.RDF4JValueExprTranslator.getTerm(RDF4JValueExprT

In [23]:
# 1. Đọc SPARQL Reference local của câu 18
original = (QUERY_DIR / "18_local.rq").read_text(encoding="utf-8")

# 2. Thay hàm data.world bằng hàm tương đương của Ontop
old_function = 'fn:date_diff(?ClaimOpenDate, ?ClaimCloseDate, "day")'
new_function = 'ofn:daysBetween(?ClaimOpenDate, ?ClaimCloseDate)'

assert original.count(old_function) == 1

query = (
    "PREFIX ofn: <http://www.ontotext.com/sparql/functions/>\n"
    + original.replace(old_function, new_function)
)

# 3. Lưu bản truy vấn tương thích
query_file = QUERY_DIR / "18_ontop.rq"
result_file = RESULT_DIR / "18_ontop.csv"

query_file.write_text(query, encoding="utf-8")
result_file.unlink(missing_ok=True)

# 4. Thực thi bằng Ontop
result = subprocess.run(
    [
        str(ONTOP_BAT), "query",
        "-m", MAPPING_FILE.resolve().as_uri(),
        "-t", str(OWL_FILE.resolve()),
        "-p", PROPERTIES_FILE.resolve().as_uri(),
        "-q", str(query_file),
        "-o", str(result_file)
    ],
    cwd=ONTOP_BAT.parent,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=180
)

# 5. Kiểm tra cả lỗi bên trong output
if (
    result.returncode != 0
    or not result_file.exists()
    or "Unexpected exception" in result.stdout
    or "QueryEvaluationException" in result.stderr
):
    print(result.stdout[-2000:])
    print(result.stderr[-2000:])
    raise RuntimeError("Question 18 failed")

# 6. Đối chiếu với SQL Reference
inquiry = inquiries[17]

sql_ref = next(
    ref for ref in benchmark.objects(inquiry, QA.expects)
    if (ref, RDF.type, DWT.SqlQuery) in benchmark
)

# Lấy SQL Reference nguyên bản
sql_query = str(benchmark.value(sql_ref, QA.queryText))

# Chuyển cú pháp DATE_DIFF của data.world sang DuckDB
sql_query_local, changes = re.subn(
    r'DATE_DIFF\(\s*claim_open_date\s*,\s*claim_close_date\s*,\s*"day"\s*\)',
    "date_diff('day', claim_open_date, claim_close_date)",
    sql_query,
    flags=re.IGNORECASE
)

assert changes == 1

# Chạy SQL Reference trên DuckDB
with duckdb.connect(str(COMPAT_DB), read_only=True) as conn:
    sql_result = conn.execute(sql_query_local).df()

sparql_result = pd.read_csv(result_file)

print("=== QUESTION 18 VALIDATION ===")
print("SQL Reference:")
display(sql_result)

print("SPARQL Reference (Ontop):")
display(sparql_result)

matched = (
    sql_result.shape == sparql_result.shape == (1, 2)
    and str(sql_result.iloc[0, 0]) == str(sparql_result.iloc[0, 0])
    and abs(
        float(sql_result.iloc[0, 1])
        - float(sparql_result.iloc[0, 1])
    ) < 1e-6
)

print("Results Match:", matched)

=== QUESTION 18 VALIDATION ===
SQL Reference:


,Policy_Number,"avg(date_diff('day', claim_open_date, claim_close_date))"
0,31003000336,20.5


SPARQL Reference (Ontop):


,PolicyNumber,AvgDaysToSettle
0,31003000336,20.5


Results Match: True


In [24]:
from collections import Counter
from decimal import Decimal

# 1. Thư mục chứa kết quả tham chiếu
SQL_REF_DIR = PROJECT_DIR / "results" / "sql" / "reference_results"

def normalize_value(value):
    value = str(value).strip()

    if value == "" or value.lower() in {"nan", "null", "none"}:
        return "NULL"

    # Chuẩn hóa số, giữ nguyên độ chính xác của số nguyên
    try:
        number = Decimal(value)
        if number.is_finite():
            if number == number.to_integral_value():
                return "N:" + str(int(number))
            return "N:" + str(round(number, 6))
    except Exception:
        pass

    # Chuẩn hóa DATE và TIMESTAMP
    if re.match(r"^\d{4}-\d{2}-\d{2}", value):
        try:
            return "D:" + pd.to_datetime(value, utc=True).isoformat()
        except Exception:
            pass

    return "S:" + value

# 2. So sánh bảng, bỏ qua thứ tự dòng và cột
def same_results(sql_df, sparql_df):
    if sql_df.shape != sparql_df.shape:
        return False

    a = [
        tuple(normalize_value(v) for v in row)
        for row in sql_df.itertuples(index=False, name=None)
    ]
    b = [
        tuple(normalize_value(v) for v in row)
        for row in sparql_df.itertuples(index=False, name=None)
    ]

    n = sql_df.shape[1]

    candidates = [
        [
            i for i in range(n)
            if Counter(row[i] for row in a)
            == Counter(row[j] for row in b)
        ]
        for j in range(n)
    ]

    order = sorted(range(n), key=lambda j: len(candidates[j]))
    mapping = {}

    def check(k):
        if k == n:
            return Counter(
                tuple(row[mapping[j]] for j in range(n))
                for row in a
            ) == Counter(b)

        j = order[k]
        for i in candidates[j]:
            if i not in mapping.values():
                mapping[j] = i
                if check(k + 1):
                    return True
                del mapping[j]
        return False

    return check(0)

# 3. Kiểm tra từng câu hỏi
comparison_log = []

for question_id, inquiry in enumerate(inquiries, start=1):
    refs = [
        ref for ref in benchmark.objects(inquiry, QA.expects)
        if (ref, RDF.type, DWT.SqlQuery) in benchmark
    ]

    sparql_file = RESULT_DIR / (
        "18_ontop.csv" if question_id == 18
        else f"{question_id:02d}.csv"
    )

    status = "MISSING"
    matched_ref = ""

    if sparql_file.exists():
        sparql_df = pd.read_csv(
            sparql_file, dtype=str, keep_default_na=False
        )

        status = "SQL_MISSING"

        for ref in refs:
            sql_file = SQL_REF_DIR / (
                str(ref).split("/")[-1] + ".csv"
            )

            if not sql_file.exists():
                continue

            sql_df = pd.read_csv(
                sql_file, dtype=str, keep_default_na=False
            )

            status = "MISMATCH"

            if same_results(sql_df, sparql_df):
                status = "MATCH"
                matched_ref = str(ref)
                break

    comparison_log.append({
        "ID": question_id,
        "Status": status,
        "SQL Reference": matched_ref,
        "SPARQL Rows": (
            len(sparql_df) if sparql_file.exists() else None
        )
    })

# 4. Tổng hợp
comparison_df = pd.DataFrame(comparison_log)

comparison_df.to_csv(
    ONTOP_DIR / "reference_comparison.csv",
    index=False
)

print("=== SQL VS SPARQL REFERENCE COMPARISON ===")
print("Total:", len(comparison_df))
print(comparison_df["Status"].value_counts())

print("\nUnmatched Questions:")
display(comparison_df[comparison_df["Status"] != "MATCH"])

=== SQL VS SPARQL REFERENCE COMPARISON ===
Total: 43
Status
MATCH    43
Name: count, dtype: int64

Unmatched Questions:


,ID,Status,SQL Reference,SPARQL Rows
